In [ ]:
from google.colab import drive
drive.mount('/content/drive')

"""
A-Z process flow for this file:
- load CSV
- build structured text strings
- clean text
- stratified split
- build label2id / id2label
- map label integers into each split
- convert to HF dataset
- tokenize with max_length=256
- remove text + target columns from dfs
- convert to torch format
- return datasets / mappings
"""

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import shutil
import os
from transformers import AutoTokenizer, DataCollatorWithPadding
from datasets import Dataset, load_from_disk, Features, Value, ClassLabel
from sklearn.model_selection import train_test_split
import sys
import json
import torch
from pathlib import Path
from torch.utils.data import Dataset as TorchDataset
from sklearn.utils.class_weight import compute_class_weight

cicids_dir = '/content/drive/MyDrive/School Archives/Wentworth Classes/Year 2/Semester 2/Thesis 2/Datasets/CICIDS2017'
toniot_dir = '/content/drive/MyDrive/School Archives/Wentworth Classes/Year 2/Semester 2/Thesis 2/Datasets/TON-IoT'
CACHE_DIR = "/content/drive/MyDrive/School Archives/Wentworth Classes/Year 2/Semester 2/Thesis 2/cache"

os.makedirs(CACHE_DIR, exist_ok=True)

def load_sensor_dataframe():
  df_all = pd.DataFrame()
  frames = []
  for root, _, files in os.walk(toniot_dir):
    for f in files:
      if f.endswith('.csv') and 'network' not in f:
        file_path = os.path.join(root, f)
        df = pd.read_csv(file_path)
        frames.append(df)

  if len(frames) > 0:
    df_all = pd.concat(frames, ignore_index=True)

  else:
    raise ValueError("No sensor files found")

  return df_all


def load_network_dataframe():
  df_all = pd.DataFrame()
  frames = []
  for root, _, files in os.walk(toniot_dir):
    for f in files:
      if f.endswith("network.csv"):
        file_path = os.path.join(root, f)
        df = pd.read_csv(file_path)
        frames.append(df)

  if len(frames) > 0:
    df_all = pd.concat(frames, ignore_index=True)

  else:
    raise ValueError("No network files found")

  return df_all


def load_cicids_dataframe():
  df_all = pd.DataFrame()
  frames = []
  for file in os.listdir(cicids_dir):
    if file.endswith(".parquet"):
      file_path = os.path.join(cicids_dir, file)
      df = pd.read_parquet(file_path)
      frames.append(df)

  if len(frames) > 0:
    df_all = pd.concat(frames, ignore_index=True)

  else:
    raise ValueError("No CICIDS files found")

  return df_all


def unify_target(df):
  for col in ["type", "Label", "label"]:
    if col in df.columns:
      df["target"] = df[col]
      df.drop(columns=[col], inplace=True, errors="ignore")
      break

  if "target" not in df.columns:
    raise ValueError("No attack label found")

  return df


def build_timestamp(df):
  if "date" in df.columns and "time" in df.columns:
    ts = pd.to_datetime(df["date"].astype(str) + " " + df["time"].astype(str), errors="coerce", format="mixed")
    ts = ts.ffill()
    ts_num = ts.astype("int64")

  elif "duration" in df.columns:
    delta_t = pd.to_numeric(df["duration"], errors="coerce").fillna(0)
    ts_num = delta_t.cumsum().astype("int64")

  elif "flow_duration" in df.columns or "Flow Duration" in df.columns:
    delta_t = (
        pd.to_numeric(df["flow_duration"], errors="coerce").fillna(0)
        if "flow_duration" in df.columns
        else pd.to_numeric(df["Flow Duration"], errors="coerce").fillna(0)
    )
    ts_num = delta_t.cumsum().astype("int64")

  elif "flow_iat_mean" in df.columns or "Flow IAT Mean" in df.columns:
    delta_t = (
        pd.to_numeric(df["flow_iat_mean"], errors="coerce").fillna(0)
        if "flow_iat_mean" in df.columns
        else pd.to_numeric(df["Flow IAT Mean"], errors="coerce").fillna(0)
    )
    ts_num = delta_t.cumsum().astype("int64")

  else:
    ts_num = np.arange(len(df), dtype=np.int64)

  return pd.DataFrame({"__ts__": ts_num})


def build_windows_text(df, window):
  df = df.sort_values("__ts__").reset_index(drop=True)
  texts = df["text"].tolist()
  labels = df["label"].tolist()

  new_texts = []
  new_labels = []

  for i in range(len(texts) - window + 1):
    new_texts.append(" <WINDOW_SEP> ".join(texts[i : i + window]))
    new_labels.append(labels[i + window - 1])

  return pd.DataFrame({"text": new_texts, "label": new_labels})


def preprocess_df(df):
  df = unify_target(df)

  # clean
  df.columns = (
      df.columns
      .str.lower()
      .str.replace(' ', '_')
  )

  cols = df.columns
  cols_to_use = [c for c in cols if c not in ["target", "__ts__"]]

  df_tmp = df[cols_to_use].copy()

  for c in df_tmp.columns:
    if pd.api.types.is_numeric_dtype(df_tmp[c]):
      df_tmp[c] = df_tmp[c].round(3).astype(str)
    else:
      df_tmp[c] = df_tmp[c].fillna("none").astype(str).str.lower()

    df_tmp[c] = c + "=" + df_tmp[c]

  df_tmp = df_tmp.replace("-", "none")
  df["text"] = df_tmp.agg(" ".join, axis=1)
  df["text"] = df["text"].str.lower()
  df["text"] = df["text"].str.replace("label=.*? ", "", regex=True)
  df = df.drop(columns=['label'], errors="ignore")

  return df


def ensure_local_copy(mode, tokenizer_name, max_length, window, cache_dir, local_dir="/content/local_cache"):
  safe_tokenizer = tokenizer_name.replace("/", "-")

  name = f"tokenized_sequence_mode_{mode}_{safe_tokenizer}_max{max_length}_w{window}" if window is not None else f"tokenized_static_mode_{mode}_{safe_tokenizer}_max{max_length}"

  src = os.path.join(cache_dir, name)

  dst = os.path.join(local_dir, name)

  done_file = os.path.join(dst, "DONE")

  src_done = os.path.join(src, "DONE")

  os.makedirs(local_dir, exist_ok=True)

  if os.path.exists(dst) and os.path.exists(done_file):
    print(f"Local cache found: {dst}")
    return dst

  # copy from drive and load
  elif os.path.exists(src) and os.path.exists(src_done):
    if os.path.exists(dst):
      print(f"Removing unneeded data from directory: {dst}")
      shutil.rmtree(dst)
    print(f"Copying {src} to {dst}")
    shutil.copytree(src, dst)
    return dst

  # missing cache, build needed
  else:
    print(f"No local cache found, will build at: {dst}")
    return dst


def load_and_prepare_data(mode: int, tokenizer_name, max_length=128):
  min_samples = 10
  cache_file = os.path.join(CACHE_DIR, f"text_raw_mode_{mode}_ms{min_samples}.parquet")
  splits_dir = os.path.join(CACHE_DIR, f"text_splits_static_mode_{mode}")
  label_map_file = os.path.join(CACHE_DIR, f"label_static_map_mode_{mode}.json")
  tokenized_dir = ensure_local_copy(mode, tokenizer_name, max_length, None, CACHE_DIR)

  tokenizer = AutoTokenizer.from_pretrained(tokenizer_name)
  if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

  def tokenize(batch):
    return tokenizer(
        batch['text'],
        padding=False,
        truncation=True,
        max_length=max_length
    )

  if os.path.exists(cache_file):
    print(f"Loading cache data from {cache_file}...")
    df = pd.read_parquet(cache_file)

  else:
    print("Building dataset from raw files...")

    # preprocess and concatenate
    if mode == 0:
      network_df = load_network_dataframe()
      cicids_df = load_cicids_dataframe()
      df_network = preprocess_df(network_df)
      df_network["__ts__"] = build_timestamp(df_network)["__ts__"]
      df_cicids = preprocess_df(cicids_df)
      df_cicids["__ts__"] = build_timestamp(df_cicids)["__ts__"]
      df = pd.concat([df_network, df_cicids], axis=0, ignore_index=True)

    elif mode == 1:
      sensor_df = load_sensor_dataframe()
      df_sensor = preprocess_df(sensor_df)
      df_sensor["__ts__"] = build_timestamp(df_sensor)["__ts__"]
      df = df_sensor

    elif mode == 2:
      network_df = load_network_dataframe()
      sensor_df = load_sensor_dataframe()
      cicids_df = load_cicids_dataframe()
      df_network = preprocess_df(network_df)
      df_network["__ts__"] = build_timestamp(df_network)["__ts__"]
      df_sensor = preprocess_df(sensor_df)
      df_sensor["__ts__"] = build_timestamp(df_sensor)["__ts__"]
      df_cicids = preprocess_df(cicids_df)
      df_cicids["__ts__"] = build_timestamp(df_cicids)["__ts__"]
      df = pd.concat([df_network, df_sensor, df_cicids], axis=0, ignore_index=True)

    counts = df["target"].value_counts()

    valid_labels = counts[counts >= min_samples].index

    df = df[df["target"].isin(valid_labels)]

    if "__ts__" in df.columns:
      df["__ts__"] = pd.to_numeric(df["__ts__"], errors="coerce")
      df = df.dropna(subset=["__ts__"])
      df["__ts__"] = df["__ts__"].astype("int64")

    df.to_parquet(cache_file)

  df = df.drop(columns=['__ts__'], errors="ignore")

  df = df.fillna('none')

  if os.path.exists(splits_dir):
    print(f"Loading splits from {splits_dir}...")

    train_df = pd.read_parquet(os.path.join(splits_dir, "train_df.parquet"))
    val_df = pd.read_parquet(os.path.join(splits_dir, "val_df.parquet"))
    test_df = pd.read_parquet(os.path.join(splits_dir, "test_df.parquet"))

  else:
    print("Splitting data from raw files...")
    # stratify
    X = df['text']
    y = df['target']
    X_tr, X_temp, y_tr, y_temp = train_test_split(X, y, test_size=0.3, stratify=y, random_state=42)
    X_val, X_te, y_val, y_te = train_test_split(X_temp, y_temp, test_size=0.5, stratify=y_temp, random_state=42)

    train_df = pd.concat([X_tr, y_tr], axis=1)
    val_df = pd.concat([X_val, y_val], axis=1)
    test_df = pd.concat([X_te, y_te], axis=1)

    os.makedirs(splits_dir, exist_ok=True)

    train_df.to_parquet(os.path.join(splits_dir, "train_df.parquet"))
    val_df.to_parquet(os.path.join(splits_dir, "val_df.parquet"))
    test_df.to_parquet(os.path.join(splits_dir, "test_df.parquet"))

  # map labels
  if os.path.exists(label_map_file):
    print(f"Loading label map from {label_map_file}...")
    with open(label_map_file, "r") as f:
      label2id = json.load(f)
    id2label = {v: k for k, v in label2id.items()}

  else:
    print("Building label map...")

    label2id = {label: idx for idx, label in enumerate(sorted(train_df['target'].unique()))}
    id2label = {idx: label for label, idx in label2id.items()}
    with open(label_map_file, "w") as f:
      json.dump(label2id, f)

  if os.path.exists(tokenized_dir) and os.path.exists(os.path.join(tokenized_dir, "DONE")):
    print(f"Loading tokenized data from {tokenized_dir}...")
    train_dataset = Dataset.load_from_disk(os.path.join(tokenized_dir, "train_dataset"))
    val_dataset = Dataset.load_from_disk(os.path.join(tokenized_dir, "val_dataset"))
    test_dataset = Dataset.load_from_disk(os.path.join(tokenized_dir, "test_dataset"))

  else:
    print("Tokenizing data...")

    train_df = train_df.copy()
    train_df['label'] = train_df['target'].map(label2id)

    val_df = val_df.copy()
    val_df['label'] = val_df['target'].map(label2id)

    test_df = test_df.copy()
    test_df['label'] = test_df['target'].map(label2id)

    train_df = train_df[['text', 'label']]
    val_df = val_df[['text', 'label']]
    test_df = test_df[['text', 'label']]

    train_dataset = Dataset.from_pandas(train_df, preserve_index=False)
    val_dataset = Dataset.from_pandas(val_df, preserve_index=False)
    test_dataset = Dataset.from_pandas(test_df, preserve_index=False)

    train_dataset = train_dataset.map(tokenize, batched=True, batch_size=128, num_proc=min(4, os.cpu_count()))
    train_dataset = train_dataset.remove_columns(['text'])
    cols_to_remove = [col for col in train_dataset.column_names if col not in ['input_ids', 'attention_mask', 'label']]
    train_dataset = train_dataset.remove_columns(cols_to_remove)

    val_dataset = val_dataset.map(tokenize, batched=True, batch_size=128, num_proc=min(4, os.cpu_count()))
    val_dataset = val_dataset.remove_columns(['text'])
    cols_to_remove = [col for col in val_dataset.column_names if col not in ['input_ids', 'attention_mask', 'label']]
    val_dataset = val_dataset.remove_columns(cols_to_remove)

    test_dataset = test_dataset.map(tokenize, batched=True, batch_size=128, num_proc=min(4, os.cpu_count()))
    test_dataset = test_dataset.remove_columns(['text'])
    cols_to_remove = [col for col in test_dataset.column_names if col not in ['input_ids', 'attention_mask', 'label']]
    test_dataset = test_dataset.remove_columns(cols_to_remove)

    train_dataset = train_dataset.rename_column('label', 'labels')
    val_dataset = val_dataset.rename_column('label', 'labels')
    test_dataset = test_dataset.rename_column('label', 'labels')

    os.makedirs(tokenized_dir, exist_ok=True)

    train_dataset.save_to_disk(os.path.join(tokenized_dir, "train_dataset"))
    val_dataset.save_to_disk(os.path.join(tokenized_dir, "val_dataset"))
    test_dataset.save_to_disk(os.path.join(tokenized_dir, "test_dataset"))

    open(os.path.join(tokenized_dir, "DONE"), "w").close()

    drive_path = os.path.join(CACHE_DIR, os.path.basename(tokenized_dir))

    tmp_path = drive_path + "_tmp"

    if not os.path.exists(os.path.join(drive_path, "DONE")):
      if os.path.exists(drive_path):
        print(f"Removing {drive_path}")
        shutil.rmtree(drive_path)
      if os.path.exists(tmp_path):
        print(f"Removing {tmp_path}")
        shutil.rmtree(tmp_path)
      print(f"Copying {tokenized_dir} to {tmp_path}")
      shutil.copytree(tokenized_dir, tmp_path)
      os.rename(tmp_path, drive_path)

  labels = np.array(train_dataset['labels'])

  class_weights = compute_class_weight(class_weight="balanced", classes=np.unique(labels), y=labels)

  class_weights = torch.tensor(class_weights, dtype=torch.float)

  train_dataset.set_format(type='torch',
                        columns=['input_ids', 'attention_mask', 'labels'])
  val_dataset.set_format(type='torch',
                        columns=['input_ids', 'attention_mask', 'labels'])
  test_dataset.set_format(type='torch',
                          columns=['input_ids', 'attention_mask', 'labels'])

  return train_dataset, val_dataset, test_dataset, label2id, id2label, class_weights


def load_and_prepare_sequence_data(mode: int, tokenizer_name, max_length=256, window=None):
  min_samples = 5
  cache_file = os.path.join(CACHE_DIR, f"text_raw_mode_{mode}_ms{min_samples}.parquet")
  splits_dir = os.path.join(CACHE_DIR, f"text_splits_sequence_mode_{mode}_w{window}")
  label_map_file = os.path.join(CACHE_DIR, f"label_sequence_map_mode_{mode}.json")
  tokenized_dir = ensure_local_copy(mode, tokenizer_name, max_length, window, CACHE_DIR)

  tokenizer = AutoTokenizer.from_pretrained(tokenizer_name)
  if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

  def tokenize(batch):
    return tokenizer(
        batch['text'],
        padding=False,
        truncation=True,
        max_length=max_length
    )

  if os.path.exists(cache_file):
    print(f"Loading cache data from {cache_file}...")
    df = pd.read_parquet(cache_file)

    if "__ts__" in df.columns:
      df = df.dropna(subset=["__ts__"])

  else:
    print("Building dataset from raw files...")

    # preprocess and concatenate
    if mode == 0:
      network_df = load_network_dataframe()
      cicids_df = load_cicids_dataframe()
      df_network = preprocess_df(network_df)
      df_network["__ts__"] = build_timestamp(df_network)["__ts__"]
      df_cicids = preprocess_df(cicids_df)
      df_cicids["__ts__"] = build_timestamp(df_cicids)["__ts__"]
      df = pd.concat([df_network, df_cicids], axis=0, ignore_index=True)

    elif mode == 1:
      sensor_df = load_sensor_dataframe()
      df_sensor = preprocess_df(sensor_df)
      df_sensor["__ts__"] = build_timestamp(df_sensor)["__ts__"]
      df = df_sensor

    elif mode == 2:
      network_df = load_network_dataframe()
      sensor_df = load_sensor_dataframe()
      cicids_df = load_cicids_dataframe()
      df_network = preprocess_df(network_df)
      df_network["__ts__"] = build_timestamp(df_network)["__ts__"]
      df_sensor = preprocess_df(sensor_df)
      df_sensor["__ts__"] = build_timestamp(df_sensor)["__ts__"]
      df_cicids = preprocess_df(cicids_df)
      df_cicids["__ts__"] = build_timestamp(df_cicids)["__ts__"]
      df = pd.concat([df_network, df_sensor, df_cicids], axis=0, ignore_index=True)

    counts = df["target"].value_counts()

    valid_labels = counts[counts >= min_samples].index

    df = df[df["target"].isin(valid_labels)]

    if "__ts__" in df.columns:
      df["__ts__"] = pd.to_numeric(df["__ts__"], errors="coerce")
      df = df.dropna(subset=["__ts__"])
      df["__ts__"] = df["__ts__"].astype("int64")

    df.to_parquet(cache_file)

  df = df.fillna('none')

  if os.path.exists(splits_dir):
    print(f"Loading splits from {splits_dir}...")
    train_df = pd.read_parquet(os.path.join(splits_dir, "train_df.parquet"))
    val_df = pd.read_parquet(os.path.join(splits_dir, "val_df.parquet"))
    test_df = pd.read_parquet(os.path.join(splits_dir, "test_df.parquet"))

  else:
    print("Splitting data from raw files...")

    train_df, temp_df = train_test_split(df, test_size=0.30, stratify=df['target'], random_state=42)
    val_df, test_df = train_test_split(temp_df, test_size=0.50, stratify=temp_df['target'], random_state=42)

    train_df = train_df[["text", "target", "__ts__"]].copy()
    val_df = val_df[["text", "target", "__ts__"]].copy()
    test_df = test_df[["text", "target", "__ts__"]].copy()

    os.makedirs(splits_dir, exist_ok=True)

    train_df.to_parquet(os.path.join(splits_dir, "train_df.parquet"))
    val_df.to_parquet(os.path.join(splits_dir, "val_df.parquet"))
    test_df.to_parquet(os.path.join(splits_dir, "test_df.parquet"))

  # map labels
  if os.path.exists(label_map_file):
    print(f"Loading label map from {label_map_file}...")
    with open(label_map_file, "r") as f:
      label2id = json.load(f)
      id2label = {v: k for k, v in label2id.items()}

  else:
    print("Building label map...")
    label2id = {label: idx for idx, label in enumerate(sorted(train_df['target'].unique()))}
    id2label = {idx: label for label, idx in label2id.items()}
    with open(label_map_file, "w") as f:
      json.dump(label2id, f)

  if os.path.exists(tokenized_dir) and os.path.exists(os.path.join(tokenized_dir, "DONE")):
    print(f"Loading tokenized data from {tokenized_dir}...")
    train_dataset = Dataset.load_from_disk(os.path.join(tokenized_dir, "train_dataset"))
    val_dataset = Dataset.load_from_disk(os.path.join(tokenized_dir, "val_dataset"))
    test_dataset = Dataset.load_from_disk(os.path.join(tokenized_dir, "test_dataset"))

  else:
    print("Tokenizing data...")

    train_df = train_df.copy()
    train_df['label'] = train_df['target'].map(label2id)
    val_df = val_df.copy()
    val_df['label'] = val_df['target'].map(label2id)
    test_df = test_df.copy()
    test_df['label'] = test_df['target'].map(label2id)

    train_df = train_df[['text', 'label', '__ts__']].reset_index(drop=True)
    val_df = val_df[['text', 'label', '__ts__']].reset_index(drop=True)
    test_df = test_df[['text', 'label', '__ts__']].reset_index(drop=True)

    # apply windows per split
    if window is not None:
      features = Features({"text": Value("string"), "label": ClassLabel(num_classes=len(label2id))})

      train_df = build_windows_text(train_df, window)
      val_df = build_windows_text(val_df, window)
      test_df = build_windows_text(test_df, window)

      if len(train_df) == 0:
        raise ValueError("Train dataset too small for window size")

      if len(val_df) == 0:
        print("Warning: Validation dataset smaller than window")

      if len(test_df) == 0:
        print("Warning: Test dataset smaller than window")

      train_dataset = Dataset.from_pandas(train_df, features=features, preserve_index=False)
      val_dataset = Dataset.from_pandas(val_df, features=features, preserve_index=False)
      test_dataset = Dataset.from_pandas(test_df, features=features, preserve_index=False)

    else:
      train_dataset = Dataset.from_pandas(train_df, preserve_index=False)
      val_dataset = Dataset.from_pandas(val_df, preserve_index=False)
      test_dataset = Dataset.from_pandas(test_df, preserve_index=False)

    train_dataset = train_dataset.map(tokenize, batched=True, batch_size=256, num_proc=min(4, os.cpu_count()))
    train_dataset = train_dataset.remove_columns(['text'])
    cols_to_remove = [col for col in train_dataset.column_names if col not in ['input_ids', 'attention_mask', 'label']]
    train_dataset = train_dataset.remove_columns(cols_to_remove)

    val_dataset = val_dataset.map(tokenize, batched=True, batch_size=256, num_proc=min(4, os.cpu_count()))
    val_dataset = val_dataset.remove_columns(['text'])
    cols_to_remove = [col for col in val_dataset.column_names if col not in ['input_ids', 'attention_mask', 'label']]
    val_dataset = val_dataset.remove_columns(cols_to_remove)

    test_dataset = test_dataset.map(tokenize, batched=True, batch_size=256, num_proc=min(4, os.cpu_count()))
    test_dataset = test_dataset.remove_columns(['text'])
    cols_to_remove = [col for col in test_dataset.column_names if col not in ['input_ids', 'attention_mask', 'label']]
    test_dataset = test_dataset.remove_columns(cols_to_remove)

    train_dataset = train_dataset.rename_column('label', 'labels')
    val_dataset = val_dataset.rename_column('label', 'labels')
    test_dataset = test_dataset.rename_column('label', 'labels')

    os.makedirs(tokenized_dir, exist_ok=True)

    train_dataset.save_to_disk(os.path.join(tokenized_dir, "train_dataset"))
    val_dataset.save_to_disk(os.path.join(tokenized_dir, "val_dataset"))
    test_dataset.save_to_disk(os.path.join(tokenized_dir, "test_dataset"))

    open(os.path.join(tokenized_dir, "DONE"), "w").close()

    drive_path = os.path.join(CACHE_DIR, os.path.basename(tokenized_dir))

    tmp_path = drive_path + "_tmp"

    if not os.path.exists(os.path.join(drive_path, "DONE")):
      if os.path.exists(drive_path):
        print(f"Removing {drive_path}")
        shutil.rmtree(drive_path)
      if os.path.exists(tmp_path):
        print(f"Removing {tmp_path}")
        shutil.rmtree(tmp_path)
      print(f"Copying {tokenized_dir} to {tmp_path}")
      shutil.copytree(tokenized_dir, tmp_path)
      os.rename(tmp_path, drive_path)

  labels = np.array(train_dataset['labels'])

  class_weights = compute_class_weight(class_weight="balanced", classes=np.unique(labels), y=labels)

  class_weights = torch.tensor(class_weights, dtype=torch.float)

  train_dataset.set_format(type='torch',
                          columns=['input_ids', 'attention_mask', 'labels'])
  val_dataset.set_format(type='torch',
                        columns=['input_ids', 'attention_mask', 'labels'])
  test_dataset.set_format(type='torch',
                          columns=['input_ids', 'attention_mask', 'labels'])

  return train_dataset, val_dataset, test_dataset, label2id, id2label, class_weights
